In [10]:
%pip install numpy matplotlib ipywidgets pandas openpyxl
%pip install scipy


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    widgets = None
    clear_output = None
    display = None

try:
    from scipy.optimize import least_squares
except ImportError as exc:
    raise ImportError("Install scipy before running this notebook: pip install scipy") from exc


In [12]:
#Settings

AFM_FOLDER = Path("afm_data")

DEFAULT_X_BOUNDARY_NM = -25.0
MIN_ELASTIC_POINTS_TO_ZERO = 100
AUTO_BOUNDARY_COARSE_CANDIDATES = 60
AUTO_BOUNDARY_REFINE_CANDIDATES = 60
POISSON_RATIO = 0.50
PYRAMID_HALF_ANGLE_DEG = 20.0

DEFAULT_X_MIN = -150
DEFAULT_X_MAX = 200

MIN_LINEAR_POINTS = 100
MIN_ELASTIC_POINTS_TO_ZERO = 100 
MIN_ELASTICITY_POINTS = 12
CURVE_OFFSET = 0


In [13]:
#Read file names and sort them 
def natural_sort_key(value):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(value))
    ]


def safe_float(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return float(number) if np.isfinite(number) else None


def extract_curve_number(name):
    match = re.search(r"cropped_(\d+)", str(name), re.IGNORECASE)
    if match is None:
        match = re.search(r"_(\d+)(?:\.[^.]+)?$", str(name), re.IGNORECASE)
    if match is None:
        return None
    return int(match.group(1)) + CURVE_OFFSET


def acquisition_from_name(name):
    text = Path(str(name)).name

    curve_match = re.match(r"(.+?)-cropped_(?:\d+)", text, re.IGNORECASE)
    if curve_match:
        return curve_match.group(1)

    tsv_match = re.match(r"(.+?)-cropped_processed-", text, re.IGNORECASE)
    if tsv_match:
        return tsv_match.group(1)

    return None


def read_header_number(file_path, key):
    pattern = re.compile(
        rf"^#\s*{re.escape(key)}\s*:\s*(.+?)\s*$",
        re.IGNORECASE,
    )

    with Path(file_path).open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            match = pattern.match(line)
            if match:
                return safe_float(match.group(1))

    return None

In [14]:
#Read AFM data from a file and return the x and y values as numpy arrays. convert to nm and nN 
# Find the exact processed TSV row for the selected TXT curve. Read actual JPK slope, kB, E and RMS values
def load_afm_txt(file_path):
    """Return the Extend and Retract segments from one AFM TXT file."""

    segments = {}
    current_segment = None

    with Path(file_path).open("r", encoding="utf-8", errors="ignore") as handle:
        for raw_line in handle:
            line = raw_line.strip()

            if line.lower().startswith("# segment:"):
                current_segment = line.split(":", 1)[1].strip().lower()
                segments.setdefault(current_segment, [])
                continue

            if not line or line.startswith("#") or current_segment is None:
                continue

            try:
                row = [float(value) for value in line.split()]
            except ValueError:
                continue

            if len(row) >= 2:
                segments[current_segment].append(row[:2])

    return (
        np.asarray(segments.get("extend", []), dtype=float),
        np.asarray(segments.get("retract", []), dtype=float),
    )


def finite_xy_nm_nN(segment):
    if segment.ndim != 2 or segment.shape[1] < 2 or segment.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    x_nm = segment[:, 0] * 1e9
    y_nN = segment[:, 1] * 1e9
    valid = np.isfinite(x_nm) & np.isfinite(y_nN)
    return x_nm[valid], y_nN[valid]

_TSV_CACHE = {}


def read_tsv(tsv_path):
    key = str(Path(tsv_path).resolve())
    if key not in _TSV_CACHE:
        _TSV_CACHE[key] = pd.read_csv(tsv_path, sep="\t")
    return _TSV_CACHE[key]


def find_exact_jpk_row(txt_path, curve_number):
    txt_path = Path(txt_path)
    acquisition = acquisition_from_name(txt_path.name)

    source = {
        "status": None,
        "folder": str(txt_path.parent),
        "tsv": None,
        "row": None,
        "filename_field": None,
        "expected_acquisition": acquisition,
        "expected_curve": curve_number,
    }

    if acquisition is None or curve_number is None:
        source["status"] = "TXT acquisition or curve number could not be parsed"
        return None, source

    local_tsvs = sorted(
        [
            path
            for path in txt_path.parent.glob("*processed*.tsv")
            if acquisition_from_name(path.name) == acquisition
        ],
        key=lambda path: natural_sort_key(path.name),
    )

    if not local_tsvs:
        source["status"] = "No processed TSV for this acquisition in the TXT folder"
        return None, source

    required_columns = {
        "Filename",
        "Slope [N/m]",
        "Young's Modulus [Pa]",
        "Contact Point [m]",
        "Baseline [N]",
        "ResidualRMS [N]",
    }

    read_errors = []

    for tsv_path in reversed(local_tsvs):
        try:
            frame = read_tsv(tsv_path)
        except Exception as exc:
            read_errors.append(f"{tsv_path.name}: {exc}")
            continue

        missing = sorted(required_columns.difference(frame.columns))
        if missing:
            read_errors.append(f"{tsv_path.name}: missing columns {missing}")
            continue

        matches = []
        for row_number, (_, row) in enumerate(frame.iterrows(), start=2):
            filename_field = str(row["Filename"])
            if (
                acquisition_from_name(filename_field) == acquisition
                and extract_curve_number(filename_field) == curve_number
            ):
                matches.append((row_number, row, filename_field))

        if len(matches) == 1:
            row_number, row, filename_field = matches[0]
            source.update(
                {
                    "status": "OK",
                    "tsv": str(tsv_path),
                    "row": row_number,
                    "filename_field": filename_field,
                }
            )
            return row, source

        if len(matches) > 1:
            source.update(
                {
                    "status": "Multiple exact rows found",
                    "tsv": str(tsv_path),
                }
            )
            return None, source

    source["status"] = "No exact acquisition and curve row found"
    if read_errors:
        source["status"] += ". " + " | ".join(read_errors)
    return None, source


def beam_stiffness_nN_per_um(kC_N_per_m, slope_N_per_m):
    if kC_N_per_m is None or slope_N_per_m is None:
        return None

    denominator = kC_N_per_m + slope_N_per_m
    if denominator == 0:
        return None

    return 1000.0 * kC_N_per_m * slope_N_per_m / denominator


def read_jpk_values(row, kC_N_per_m):
    values = {
        "kC_N_per_m": kC_N_per_m,
        "s_N_per_m": None,
        "kB_nN_per_um": None,
        "E_MPa": None,
        "rms_saved_pN": None,
        "r2": None,
        "contact_nm": None,
        "baseline_pN": None,
    }

    if row is None:
        return values

    slope = safe_float(row["Slope [N/m]"])
    slope = None if slope is None else abs(slope)
    modulus_Pa = safe_float(row["Young's Modulus [Pa]"])
    contact_m = safe_float(row["Contact Point [m]"])
    baseline_N = safe_float(row["Baseline [N]"])
    rms_N = safe_float(row["ResidualRMS [N]"])

    values.update(
        {
            "s_N_per_m": slope,
            "kB_nN_per_um": beam_stiffness_nN_per_um(kC_N_per_m, slope),
            "E_MPa": None if modulus_Pa is None else modulus_Pa / 1e6,
            "rms_saved_pN": None if rms_N is None else rms_N * 1e12,
            "contact_nm": None if contact_m is None else contact_m * 1e9,
            "baseline_pN": None if baseline_N is None else baseline_N * 1e12,
        }
    )
    return values

In [15]:
#Linear fit 
def calculate_metrics(observed_nN, predicted_nN):
    observed_nN = np.asarray(observed_nN, dtype=float)
    predicted_nN = np.asarray(predicted_nN, dtype=float)
    residual_nN = observed_nN - predicted_nN

    rms_pN = float(np.sqrt(np.mean(residual_nN**2)) * 1000.0)
    ss_res = float(np.sum(residual_nN**2))
    ss_tot = float(np.sum((observed_nN - np.mean(observed_nN)) ** 2))

    if ss_tot <= np.finfo(float).eps:
        r2 = None
    else:
        r2 = float(1.0 - ss_res / ss_tot)

    return r2, rms_pN


def fit_linear_region(x_nm, y_nN, kC_N_per_m, boundary_nm):
    mask = np.isfinite(x_nm) & np.isfinite(y_nN) & (x_nm <= boundary_nm)
    point_count = int(np.count_nonzero(mask))

    result = {
        "status": None,
        "point_count": point_count,
        "s_N_per_m": None,
        "kB_nN_per_um": None,
        "intercept_pN": None,
        "r2": None,
        "rms_pN": None,
        "x_nm": np.array([], dtype=float),
        "y_nN": np.array([], dtype=float),
    }

    if point_count < MIN_LINEAR_POINTS:
        result["status"] = f"Need at least {MIN_LINEAR_POINTS} linear points"
        return result

    selected_x_m = x_nm[mask] * 1e-9
    selected_y_N = y_nN[mask] * 1e-9

    if np.ptp(selected_x_m) <= np.finfo(float).eps:
        result["status"] = "Linear x range is zero"
        return result

    slope_N_per_m, intercept_N = np.polyfit(selected_x_m, selected_y_N, 1)
    predicted_nN = (slope_N_per_m * selected_x_m + intercept_N) * 1e9
    r2, rms_pN = calculate_metrics(y_nN[mask], predicted_nN)

    plot_x_nm = np.linspace(float(np.min(x_nm[mask])), float(np.max(x_nm[mask])), 160)
    plot_y_nN = (slope_N_per_m * plot_x_nm * 1e-9 + intercept_N) * 1e9
    slope_abs = abs(float(slope_N_per_m))

    result.update(
        {
            "status": "OK",
            "s_N_per_m": slope_abs,
            "kB_nN_per_um": beam_stiffness_nN_per_um(kC_N_per_m, slope_abs),
            "intercept_pN": float(intercept_N * 1e12),
            "r2": r2,
            "rms_pN": rms_pN,
            "x_nm": plot_x_nm,
            "y_nN": plot_y_nN,
        }
    )
    return result

In [16]:
#Elasticity fit
def elasticity_model_nN(x_nm, log_E_Pa, contact_nm, baseline_nN):
    E_Pa = np.exp(log_E_Pa)
    alpha_rad = np.deg2rad(PYRAMID_HALF_ANGLE_DEG)

    coefficient = (
        0.7453
        * E_Pa
        / (1.0 - POISSON_RATIO**2)
        * np.tan(alpha_rad)
    )

    indentation_m = np.maximum(contact_nm - np.asarray(x_nm), 0.0) * 1e-9
    return baseline_nN + coefficient * indentation_m**2 * 1e9


def fit_elasticity_region(x_nm, y_nN, boundary_nm):
    finite = np.isfinite(x_nm) & np.isfinite(y_nN)
    mask = finite & (x_nm >= boundary_nm)
    point_count = int(np.count_nonzero(mask))
    point_count_to_contact = int(
        np.count_nonzero(finite & (x_nm >= boundary_nm) & (x_nm <= 0.0))
    )

    result = {
        "status": None,
        "point_count": point_count,
        "point_count_to_contact": point_count_to_contact,
        "E_MPa": None,
        "contact_nm": None,
        "baseline_pN": None,
        "r2": None,
        "rms_pN": None,
        "x_nm": np.array([], dtype=float),
        "y_nN": np.array([], dtype=float),
    }

    if point_count_to_contact < MIN_ELASTIC_POINTS_TO_ZERO:
        result["status"] = (
            f"Need at least {MIN_ELASTIC_POINTS_TO_ZERO} elasticity points "
            "from boundary to x=0"
        )
        return result

    if point_count < MIN_ELASTICITY_POINTS:
        result["status"] = f"Need at least {MIN_ELASTICITY_POINTS} elasticity points"
        return result

    selected_x_nm = x_nm[mask]
    selected_y_nN = y_nN[mask]
    x_span = max(float(np.ptp(selected_x_nm)), 1.0)

    right_side = selected_y_nN[
        selected_x_nm >= np.quantile(selected_x_nm, 0.75)
    ]
    baseline_start = float(
        np.median(right_side if right_side.size else selected_y_nN)
    )
    contact_target = baseline_start + 0.02
    contact_start = float(
        selected_x_nm[np.argmin(np.abs(selected_y_nN - contact_target))]
    )

    y_margin = max(float(np.ptp(selected_y_nN)), 0.5)
    lower_bounds = [
        np.log(1e2),
        float(np.min(selected_x_nm) - 0.5 * x_span),
        float(np.min(selected_y_nN) - y_margin),
    ]
    upper_bounds = [
        np.log(1e10),
        float(np.max(selected_x_nm) + 0.5 * x_span),
        float(np.max(selected_y_nN) + y_margin),
    ]
    initial = [
        np.log(1e6),
        np.clip(contact_start, lower_bounds[1], upper_bounds[1]),
        np.clip(baseline_start, lower_bounds[2], upper_bounds[2]),
    ]

    def residual(parameters):
        return (
            elasticity_model_nN(selected_x_nm, *parameters)
            - selected_y_nN
        )

    try:
        solution = least_squares(
            residual,
            initial,
            bounds=(lower_bounds, upper_bounds),
            max_nfev=5000,
            x_scale="jac",
        )
    except Exception as exc:
        result["status"] = f"Elasticity optimizer failed: {exc}"
        return result

    if not solution.success or not np.all(np.isfinite(solution.x)):
        result["status"] = f"Elasticity optimizer failed: {solution.message}"
        return result

    log_E_Pa, contact_nm, baseline_nN = solution.x
    predicted_nN = elasticity_model_nN(
        selected_x_nm,
        log_E_Pa,
        contact_nm,
        baseline_nN,
    )
    r2, rms_pN = calculate_metrics(selected_y_nN, predicted_nN)

    plot_x_nm = np.linspace(
        float(np.min(selected_x_nm)),
        float(np.max(selected_x_nm)),
        220,
    )
    plot_y_nN = elasticity_model_nN(
        plot_x_nm,
        log_E_Pa,
        contact_nm,
        baseline_nN,
    )

    result.update(
        {
            "status": "OK",
            "E_MPa": float(np.exp(log_E_Pa) / 1e6),
            "contact_nm": float(contact_nm),
            "baseline_pN": float(baseline_nN * 1000.0),
            "r2": r2,
            "rms_pN": rms_pN,
            "x_nm": plot_x_nm,
            "y_nN": plot_y_nN,
        }
    )
    return result

In [17]:
#Test boundary conditions. For each one, it compares our fitted kB, E and elasticity RMS with the saved JPK values. The boundary with the lowest average percentage difference is selected. 
def _relative_error_percent(actual, fitted):
    if actual is None or fitted is None:
        return None
    try:
        actual = float(actual)
        fitted = float(fitted)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(actual) or not np.isfinite(fitted) or actual == 0:
        return None
    return abs(fitted - actual) / abs(actual) * 100.0


def _candidate_boundary_values(x_nm, y_nN):
    finite = np.isfinite(x_nm) & np.isfinite(y_nN)
    finite_x = np.sort(np.asarray(x_nm[finite], dtype=float))
    if finite_x.size == 0:
        return np.array([], dtype=float)

    possible = np.unique(finite_x[finite_x <= 0.0])
    if possible.size == 0:
        return np.array([], dtype=float)

    zero_right = np.searchsorted(finite_x, 0.0, side="right")
    feasible = []
    for boundary_nm in possible:
        left = np.searchsorted(finite_x, boundary_nm, side="left")
        right = np.searchsorted(finite_x, boundary_nm, side="right")
        linear_count = right
        elastic_count = finite_x.size - left
        to_zero_count = zero_right - left
        if (
            linear_count >= MIN_LINEAR_POINTS
            and elastic_count >= MIN_ELASTICITY_POINTS
            and to_zero_count >= MIN_ELASTIC_POINTS_TO_ZERO
        ):
            feasible.append(float(boundary_nm))

    return np.asarray(feasible, dtype=float)


def _sample_evenly(values, count):
    values = np.asarray(values, dtype=float)
    if values.size <= count:
        return values
    indices = np.unique(np.linspace(0, values.size - 1, count).round().astype(int))
    return values[indices]


def optimize_fit_boundary(x_nm, y_nN, kC_N_per_m, jpk):
    """Choose one x boundary that best reproduces saved JPK kB, E and RMS."""
    feasible = _candidate_boundary_values(x_nm, y_nN)

    info = {
        "status": None,
        "score_pct": None,
        "evaluated": 0,
        "feasible_count": int(feasible.size),
    }

    if feasible.size == 0:
        boundary_nm = DEFAULT_X_BOUNDARY_NM
        linear = fit_linear_region(x_nm, y_nN, kC_N_per_m, boundary_nm)
        elasticity = fit_elasticity_region(x_nm, y_nN, boundary_nm)
        info["status"] = "No boundary satisfied the point-count constraints"
        return boundary_nm, linear, elasticity, info

    target_kB = jpk.get("kB_nN_per_um")
    target_s = jpk.get("s_N_per_m")
    target_E = jpk.get("E_MPa")
    target_rms = jpk.get("rms_saved_pN")

    cache = {}

    def evaluate(boundary_nm):
        key = float(boundary_nm)
        if key in cache:
            return cache[key]

        linear = fit_linear_region(x_nm, y_nN, kC_N_per_m, key)
        elasticity = fit_elasticity_region(x_nm, y_nN, key)
        info["evaluated"] += 1

        if linear["status"] != "OK" or elasticity["status"] != "OK":
            result = (np.inf, linear, elasticity)
            cache[key] = result
            return result

        errors = []

        linear_error = _relative_error_percent(target_kB, linear["kB_nN_per_um"])
        if linear_error is None:
            linear_error = _relative_error_percent(target_s, linear["s_N_per_m"])
        if linear_error is not None:
            errors.append(linear_error)

        e_error = _relative_error_percent(target_E, elasticity["E_MPa"])
        if e_error is not None:
            errors.append(e_error)

        rms_error = _relative_error_percent(target_rms, elasticity["rms_pN"])
        if rms_error is not None:
            errors.append(rms_error)

        score = float(np.mean(errors)) if errors else np.inf
        result = (score, linear, elasticity)
        cache[key] = result
        return result

    coarse = _sample_evenly(feasible, AUTO_BOUNDARY_COARSE_CANDIDATES)
    if np.any(np.isclose(feasible, DEFAULT_X_BOUNDARY_NM)):
        coarse = np.unique(np.append(coarse, DEFAULT_X_BOUNDARY_NM))

    best_boundary = None
    best_score = np.inf
    best_linear = None
    best_elasticity = None

    for boundary_nm in coarse:
        score, linear, elasticity = evaluate(boundary_nm)
        if score < best_score:
            best_boundary = float(boundary_nm)
            best_score = score
            best_linear = linear
            best_elasticity = elasticity

    if best_boundary is not None:
        best_index = int(np.argmin(np.abs(feasible - best_boundary)))
        coarse_step = max(1, int(np.ceil(feasible.size / AUTO_BOUNDARY_COARSE_CANDIDATES)))
        radius = max(4, coarse_step * 2)
        lo = max(0, best_index - radius)
        hi = min(feasible.size, best_index + radius + 1)
        refine_pool = feasible[lo:hi]
        refine = _sample_evenly(refine_pool, AUTO_BOUNDARY_REFINE_CANDIDATES)

        for boundary_nm in refine:
            score, linear, elasticity = evaluate(boundary_nm)
            if score < best_score:
                best_boundary = float(boundary_nm)
                best_score = score
                best_linear = linear
                best_elasticity = elasticity

    if best_boundary is None:
        boundary_nm = float(feasible[np.argmin(np.abs(feasible - DEFAULT_X_BOUNDARY_NM))])
        linear = fit_linear_region(x_nm, y_nN, kC_N_per_m, boundary_nm)
        elasticity = fit_elasticity_region(x_nm, y_nN, boundary_nm)
        info["status"] = "Targets unavailable, used nearest feasible boundary to default"
        return boundary_nm, linear, elasticity, info

    info["status"] = "OK"
    info["score_pct"] = best_score
    return best_boundary, best_linear, best_elasticity, info

In [18]:
#Display curve one by one in form of dropdown browser. 

def analyze_curve(txt_path):
    txt_path = Path(txt_path)
    approach, retract = load_afm_txt(txt_path)
    approach_x_nm, approach_y_nN = finite_xy_nm_nN(approach)
    retract_x_nm, retract_y_nN = finite_xy_nm_nN(retract)

    if approach_x_nm.size == 0:
        raise ValueError(f"No Extend data found in {txt_path}")

    curve_number = extract_curve_number(txt_path.name)
    kC_N_per_m = read_header_number(txt_path, "springConstant")

    jpk_row, jpk_source = find_exact_jpk_row(txt_path, curve_number)
    jpk = read_jpk_values(jpk_row, kC_N_per_m)

    fit_boundary_nm, linear, elasticity, boundary_search = optimize_fit_boundary(
        approach_x_nm,
        approach_y_nN,
        kC_N_per_m,
        jpk,
    )

    return {
        "file": txt_path,
        "curve_number": curve_number,
        "approach_x_nm": approach_x_nm,
        "approach_y_nN": approach_y_nN,
        "retract_x_nm": retract_x_nm,
        "retract_y_nN": retract_y_nN,
        "jpk": jpk,
        "jpk_source": jpk_source,
        "fit_boundary_nm": fit_boundary_nm,
        "boundary_search": boundary_search,
        "linear": linear,
        "elasticity": elasticity,
    }




def format_number(value, digits=5):
    if value is None:
        return "N/A"
    try:
        number = float(value)
    except (TypeError, ValueError):
        return "N/A"
    if not np.isfinite(number):
        return "N/A"
    return f"{number:.{digits}f}"


def percentage_difference_from_actual(actual, fitted):
    """Return absolute percentage difference using the JPK value as reference."""
    if actual is None or fitted is None:
        return None
    try:
        actual = float(actual)
        fitted = float(fitted)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(actual) or not np.isfinite(fitted) or actual == 0:
        return None
    return abs(fitted - actual) / abs(actual) * 100.0


class AFMPlotBrowser:
    def __init__(self, afm_folder=AFM_FOLDER):
        if widgets is None:
            raise ImportError(
                "The dropdown browser needs ipywidgets. "
                "Run this in a notebook cell: %pip install ipywidgets"
            )

        self.afm_folder = Path(afm_folder)
        self.folders = self._scan_folders()
        self.cache = {}
        self._updating = False

        folder_options = [
            (f"{folder_name} ({len(self.folders[folder_name])} curves)", folder_name)
            for folder_name in self.folders
        ]

        self.folder_dropdown = widgets.Dropdown(
            options=folder_options,
            description="Folder:",
            layout=widgets.Layout(width="420px"),
            style={"description_width": "70px"},
        )
        self.curve_dropdown = widgets.Dropdown(
            options=[],
            description="Curve:",
            layout=widgets.Layout(width="620px"),
            style={"description_width": "70px"},
        )
        self.previous_button = widgets.Button(
            description="Previous",
            icon="arrow-left",
            layout=widgets.Layout(width="110px"),
        )
        self.next_button = widgets.Button(
            description="Next",
            icon="arrow-right",
            layout=widgets.Layout(width="110px"),
        )
        self.counter = widgets.Label()
        self.status = widgets.Label()
        self.plot_output = widgets.Output(
            layout=widgets.Layout(width="72%")
        )
        self.values_output = widgets.Output(
            layout=widgets.Layout(width="28%", min_width="330px")
        )

        self.folder_dropdown.observe(self._folder_changed, names="value")
        self.curve_dropdown.observe(self._curve_changed, names="value")
        self.previous_button.on_click(self._previous_curve)
        self.next_button.on_click(self._next_curve)

        if self.folders:
            self._load_folder(self.folder_dropdown.value)

    def _scan_folders(self):
        if not self.afm_folder.exists():
            raise FileNotFoundError(
                f"AFM folder not found: {self.afm_folder.resolve()}"
            )

        folder_map = {}
        for txt_path in self.afm_folder.rglob("*.txt"):
            folder_name = str(txt_path.parent.relative_to(self.afm_folder))
            folder_map.setdefault(folder_name, []).append(txt_path)

        if not folder_map:
            raise FileNotFoundError(
                f"No AFM TXT files found inside {self.afm_folder.resolve()}"
            )

        sorted_map = {}
        for folder_name in sorted(folder_map, key=natural_sort_key):
            sorted_map[folder_name] = sorted(
                folder_map[folder_name],
                key=lambda path: (
                    extract_curve_number(path.name) is None,
                    extract_curve_number(path.name) or 0,
                    natural_sort_key(path.name),
                ),
            )
        return sorted_map

    def _load_folder(self, folder_name):
        if folder_name is None:
            return

        self._updating = True
        files = self.folders[folder_name]
        self.curve_dropdown.options = [
            (
                f"_{extract_curve_number(path.name):03d} | {path.name}"
                if extract_curve_number(path.name) is not None
                else path.name,
                str(path),
            )
            for path in files
        ]
        if files:
            self.curve_dropdown.value = str(files[0])
        self._updating = False
        self._render_selected_curve()

    def _folder_changed(self, change):
        if change["new"] is not None:
            self._load_folder(change["new"])

    def _curve_changed(self, change):
        if not self._updating and change["new"] is not None:
            self._render_selected_curve()

    def _previous_curve(self, _button):
        values = [value for _label, value in self.curve_dropdown.options]
        if not values:
            return
        index = values.index(self.curve_dropdown.value)
        self.curve_dropdown.value = values[max(0, index - 1)]

    def _next_curve(self, _button):
        values = [value for _label, value in self.curve_dropdown.options]
        if not values:
            return
        index = values.index(self.curve_dropdown.value)
        self.curve_dropdown.value = values[min(len(values) - 1, index + 1)]

    def _get_analysis(self, file_path):
        key = str(Path(file_path).resolve())
        if key not in self.cache:
            self.status.value = f"Fitting {Path(file_path).name}..."
            self.cache[key] = analyze_curve(file_path)
        return self.cache[key]

    def _render_selected_curve(self):
        file_path = self.curve_dropdown.value
        if file_path is None:
            return

        try:
            analysis = self._get_analysis(file_path)
        except Exception as exc:
            self.status.value = f"Error: {exc}"
            with self.plot_output:
                clear_output(wait=True)
                print(f"Could not plot {file_path}\n{exc}")
            with self.values_output:
                clear_output(wait=True)
            return

        values = [value for _label, value in self.curve_dropdown.options]
        selected_index = values.index(self.curve_dropdown.value)
        self.counter.value = f"Curve {selected_index + 1} of {len(values)}"
        self.status.value = f"Showing {Path(file_path).name}"

        self._draw_plot(analysis)
        self._show_values(analysis)

    def _draw_plot(self, analysis):
        linear = analysis["linear"]
        elasticity = analysis["elasticity"]

        with self.plot_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10.5, 6.6))

            ax.plot(
                analysis["approach_x_nm"],
                analysis["approach_y_nN"],
                color="#1f77b4",
                linewidth=1.2,
                label="Approach",
            )

            if analysis["retract_x_nm"].size:
                ax.plot(
                    analysis["retract_x_nm"],
                    analysis["retract_y_nN"],
                    color="#ff7f0e",
                    linewidth=1.2,
                    label="Retract",
                )

            if linear["status"] == "OK":
                ax.plot(
                    linear["x_nm"],
                    linear["y_nN"],
                    color="#16823c",
                    linewidth=3.0,
                    label="Linear fit, our window",
                )

            if elasticity["status"] == "OK":
                ax.plot(
                    elasticity["x_nm"],
                    elasticity["y_nN"],
                    color="#d62728",
                    linewidth=3.0,
                    label="Elasticity fit, our window",
                )
                ax.axvline(
                    elasticity["contact_nm"],
                    color="#7b8794",
                    linewidth=1.0,
                    linestyle=":",
                    label="Fitted contact point",
                )

            boundary_nm = analysis["fit_boundary_nm"]
            ax.axvline(
                boundary_nm,
                color="#7b8794",
                linewidth=1.0,
                linestyle="--",
                label=f"{boundary_nm:.2f} nm auto fit boundary",
            )

            folder_name = str(
                analysis["file"].parent.relative_to(self.afm_folder)
            )
            curve_label = (
                f"_{analysis['curve_number']:03d}"
                if analysis["curve_number"] is not None
                else analysis["file"].stem
            )

            ax.set_title(f"{folder_name} | {curve_label}")
            ax.set_xlabel("Raw tip position (nm)")
            ax.set_ylabel("Force (nN)")
            ax.set_xlim(DEFAULT_X_MIN, DEFAULT_X_MAX)
            ax.grid(True, alpha=0.25)
            ax.legend(loc="best", fontsize=9)
            fig.tight_layout()
            plt.show()
            plt.close(fig)

    def _show_values(self, analysis):
        jpk = analysis["jpk"]
        source = analysis["jpk_source"]
        linear = analysis["linear"]
        elasticity = analysis["elasticity"]
        boundary_nm = analysis["fit_boundary_nm"]
        boundary_search = analysis["boundary_search"]

        with self.values_output:
            clear_output(wait=True)

            print("ACTUAL JPK, SAVED IN TSV")
            print(f"kC              {format_number(jpk['kC_N_per_m'], 8)} N/m")
            print(f"s               {format_number(jpk['s_N_per_m'], 6)} N/m")
            print(f"kB              {format_number(jpk['kB_nN_per_um'], 4)} nN/um")
            print(f"E               {format_number(jpk['E_MPa'], 5)} MPa")
            print(f"RMS saved       {format_number(jpk['rms_saved_pN'], 4)} pN")

            print("\nOUR INDEPENDENT FIT")
            print(f"Linear s        {format_number(linear['s_N_per_m'], 6)} N/m")
            print(f"Linear kB       {format_number(linear['kB_nN_per_um'], 4)} nN/um")
            print(f"Linear R2       {format_number(linear['r2'], 5)}")
            print(f"Linear RMS      {format_number(linear['rms_pN'], 4)} pN")
            print(f"Elasticity E    {format_number(elasticity['E_MPa'], 5)} MPa")
            print(f"Fit contact     {format_number(elasticity['contact_nm'], 5)} nm")
            print(f"Fit baseline    {format_number(elasticity['baseline_pN'], 5)} pN")
            print(f"Elasticity R2   {format_number(elasticity['r2'], 5)}")
            print(f"Elasticity RMS  {format_number(elasticity['rms_pN'], 4)} pN")

            linear_difference_pct = percentage_difference_from_actual(
                jpk["kB_nN_per_um"], linear["kB_nN_per_um"]
            )
            elastic_difference_pct = percentage_difference_from_actual(
                jpk["E_MPa"], elasticity["E_MPa"]
            )
            rms_difference_pct = percentage_difference_from_actual(
                jpk["rms_saved_pN"], elasticity["rms_pN"]
            )

            print("\nSANITY CHECK")
            print(f"Linear kB       {format_number(linear_difference_pct, 2)} %")
            print(f"Elastic E       {format_number(elastic_difference_pct, 2)} %")
            print(f"Elastic RMS     {format_number(rms_difference_pct, 2)} %")

            print("\nFIT WINDOWS")
            print(f"Auto boundary   {format_number(boundary_nm, 3)} nm")
            print(f"Linear          x <= {boundary_nm:.3f} nm to -infinity")
            print(f"Elasticity      x >= {boundary_nm:.3f} nm to +infinity")
            print(f"Linear points   {linear['point_count']}")
            print(f"Elastic points  {elasticity['point_count']}")
            print(
                f"Elastic points till contact point  "
                f"{elasticity['point_count_to_contact']}"
            )
            print(f"Linear status   {linear['status']}")
            print(f"Elastic status  {elasticity['status']}")
            print(f"Auto-fit status {boundary_search['status']}")
            print(
                f"Auto-fit score  "
                f"{format_number(boundary_search['score_pct'], 3)} %"
            )
            print(f"Boundaries tried {boundary_search['evaluated']}")

            print("\nEXACT JPK SOURCE")
            print(f"Match           {source['status']}")
            print(f"Folder          {source['folder']}")
            print(f"TSV             {source['tsv'] or 'N/A'}")
            print(f"TSV row         {source['row'] or 'N/A'}")
            print(f"Filename field  {source['filename_field'] or 'N/A'}")

    def display(self):
        controls = widgets.VBox(
            [
                widgets.HBox([self.folder_dropdown, self.counter]),
                widgets.HBox(
                    [
                        self.curve_dropdown,
                        self.previous_button,
                        self.next_button,
                    ]
                ),
                self.status,
            ]
        )
        content = widgets.HBox(
            [self.plot_output, self.values_output],
            layout=widgets.Layout(width="100%", align_items="flex-start"),
        )
        display(widgets.VBox([controls, content]))



def launch_afm_browser(afm_folder=AFM_FOLDER):
    """Create and display the notebook AFM plot browser."""
    browser = AFMPlotBrowser(afm_folder)
    browser.display()
    return browser


afm_browser = launch_afm_browser()

